# Week 9, Day 5: Matrix calculus: gradients of vector and matrix expressions

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w009/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Check the identities numerically

Write `grad_quad(w)`, the gradient of $\mathbf{w}^\top A\mathbf{w}$ for the non-symmetric matrix `A`, and `grad_ls(w)`, the gradient of $\frac{1}{2}\lVert X\mathbf{w} - \mathbf{y}\rVert^2$. Check both against `numerical_gradient`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1982)
A = rng.normal(size=(4, 4))               # not symmetric
X = rng.normal(size=(30, 4)); y = rng.normal(size=30)
w0 = rng.normal(size=4)

def numerical_gradient(f, w, h=1e-6):
    g = np.zeros_like(w)
    for i in range(len(w)):
        e = np.zeros_like(w); e[i] = h
        g[i] = (f(w + e) - f(w - e)) / (2 * h)
    return g

def grad_quad(w):
    return ...

def grad_ls(w):
    return ...

print(grad_quad(w0).round(4)); print(numerical_gradient(lambda v: v @ A @ v, w0).round(4))

In [ ]:
import numpy as np
def test_quad():
    "grad_quad matches the numerical gradient of w^T A w for a non-symmetric A"
    assert np.allclose(grad_quad(w0), numerical_gradient(lambda v: v @ A @ v, w0), atol=1e-5)

def test_ls():
    "grad_ls matches the numerical gradient of the squared error"
    assert np.allclose(grad_ls(w0), numerical_gradient(lambda v: 0.5 * np.sum((X @ v - y) ** 2), w0), atol=1e-5)

test_quad()
test_ls()
print("All checks passed")

### Exercise: Fit a Poisson GLM by gradient ascent

Write `loglik(w)` and `grad_ll(w)` for the Poisson GLM. Fit it by gradient ascent (w goes up the gradient, step 0.5 times the gradient divided by the number of bins) for 1000 steps from zero, recording the log-likelihood in `lls`, and compare the fitted temporal filter `w_fit` with the true one.

In [ ]:
T, d = 3000, 12
lags = np.arange(d)
filt = np.exp(-lags / 2.5) * np.sin(lags / 1.5)                    # the neuron's true temporal filter
stim = rng.normal(size=T + d)
Xg = np.column_stack([np.array([stim[t:t + d][::-1] for t in range(T)]), np.ones(T)])   # recent stimulus history plus a bias
w_true = np.append(0.6 * filt, np.log(0.3))
counts = rng.poisson(np.exp(Xg @ w_true))

def loglik(w):
    return ...

def grad_ll(w):
    return ...

w_fit = np.zeros(d + 1); lls = []
for _ in range(1000):
    w_fit = ...
    lls.append(loglik(w_fit))

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(lls); ax[0].set_xlabel("step"); ax[0].set_ylabel("log-likelihood")
ax[1].plot(lags, w_true[:-1], "o-", label="true filter"); ax[1].plot(lags, w_fit[:-1], "s--", label="fitted")
ax[1].set_xlabel("lag (bins)"); ax[1].legend(frameon=False); plt.tight_layout(); plt.show()
print("correlation of filters:", np.corrcoef(w_fit[:-1], w_true[:-1])[0, 1].round(4), " baseline rate:", np.exp(w_fit[-1]).round(3), "spikes per bin")

In [ ]:
import numpy as np
def test_grad():
    "grad_ll matches a numerical gradient of the log-likelihood"
    w = np.random.default_rng(0).normal(size=Xg.shape[1]) * 0.1
    assert np.allclose(grad_ll(w), numerical_gradient(loglik, w), rtol=1e-4, atol=1e-4)

def test_rise():
    "the log-likelihood increases over training"
    assert lls[-1] > lls[0]

def test_recover():
    "the fitted filter correlates with the true one above 0.95"
    assert np.corrcoef(w_fit[:-1], w_true[:-1])[0, 1] > 0.95

test_grad()
test_rise()
test_recover()
print("All checks passed")

### Exercise: A neuron that does PCA

Write `oja_step(w, x, eta)`. Present 5000 inputs drawn from the correlated 10-dimensional distribution below, one at a time, starting from a random unit vector with learning rate 0.005. Compare the final `w` with `top_pc`, the top eigenvector of the input covariance.

In [ ]:
dim = 10
Q = np.linalg.qr(rng.normal(size=(dim, dim)))[0]                   # random orthogonal axes
sd = np.array([2.0, 1.2, 1.0, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2])  # spread of the inputs along each axis
inputs = (rng.normal(size=(5000, dim)) * sd) @ Q.T                # correlated inputs
C = np.cov(inputs, rowvar=False)
top_pc = np.linalg.eigh(C)[1][:, -1]

def oja_step(w, x, eta):
    y_out = w @ x
    return ...

w = rng.normal(size=dim); w /= np.linalg.norm(w)
align = []
for x in inputs:
    w = oja_step(w, x, 0.005)
    align.append(abs(w @ top_pc) / np.linalg.norm(w))

plt.figure(figsize=(5, 3)); plt.plot(align); plt.xlabel("input number"); plt.ylabel("|cos(w, top PC)|"); plt.show()
print("length of w:", np.linalg.norm(w).round(3), "  alignment with the top PC:", round(abs(w @ top_pc) / np.linalg.norm(w), 4))

In [ ]:
import numpy as np
def test_update():
    "oja_step applies w + eta y (x - y w)"
    w = np.array([0.6, 0.8]); x = np.array([1.0, 2.0])
    yv = w @ x
    assert np.allclose(oja_step(w, x, 0.1), w + 0.1 * yv * (x - yv * w))

def test_converge():
    "after learning, w has unit length and lines up with the top eigenvector of the input covariance"
    assert abs(np.linalg.norm(w) - 1) < 0.05
    assert abs(w @ top_pc) > 0.99

test_update()
test_converge()
print("All checks passed")